# Create vectorstore of the data and try the model

In [19]:
import pandas as pd
from langchain_openai import OpenAIEmbeddings
from google import genai
import os
import getpass
from langchain_community.document_loaders import DataFrameLoader
from langchain_community.vectorstores import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA


In [3]:

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

Load the dataframe and inspect it

In [4]:
df = pd.read_csv("transcripts_clean.csv")

In [5]:
loader = DataFrameLoader(df,'text')
docs = loader.load()

Create the Chroma Database with the embeddings.

In [8]:
# RUN ONCE TO CREATE DB
vectorstore = Chroma.from_documents(
    docs,
    embeddings,
    collection_name="podcast-transcripts",
    persist_directory="./chroma_db"
)
vectorstore.persist()

/var/folders/kv/zhym5mj14d93r5br7y2sy5j00000gn/T/ipykernel_22388/2329692783.py:7: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [9]:
# LOAD EXISTING DB
vectorstore = Chroma(
    collection_name="podcast-transcripts",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

/var/folders/kv/zhym5mj14d93r5br7y2sy5j00000gn/T/ipykernel_22388/4007951346.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [27]:
# Create retriever from vectorstore
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [37]:
rag_template = """
You are a helpful assistant that answers questions based on the context.

Use the context below to answer the user question. 
Only use information from the context — do not invent facts.

If the answer is not in the context, say "I don’t know based on the provided context."

Context:
{context}

Question:
{question}

Answer in a clear and concise way:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=rag_template,
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)



In [38]:
result = qa_chain({"query": "do hard things make you happier?"})

print("Answer:", result["result"])  # model’s answer
print("\nSources:")
for doc in result["source_documents"]:
    print("-", doc.metadata, "\n", doc.page_content[:200], "...\n")


Answer: I don’t know based on the provided context.

Sources:
- {} 
 go, oh, I do hard things. ...

- {} 
 I've done hard things. ...

- {} 
 happiness true happiness do you think ...

